# Gold v2 — 모순 지표 + 새 위치 문항(MMAD 방식) 평가

**A. 모순 지표 (새 추론 없음, CPU로도 가능)**
기존 모든 조건(원본만 / reference 3종 / vision expert 20종 / 결합)의 문항별 예측에서 계산합니다.
- Consistency Error: Q1·Q2 답이 모순인 이미지 비율 (Q1 정상인데 Q2 결함 / Q1 이상인데 Q2 정상)
- Joint Accuracy: Q1·Q2 둘 다 맞힌 이미지 비율
- False Defect Rate: 정상 이미지에 결함 종류를 답한 비율
- 같은 행에 Q1 F1·Q2 macro-F1도 나오니, 기존 리더보드 숫자와 같은지 확인 가능

**B. 새 위치 문항 평가 (L4 필요)** — 박스 표시 노트북으로 만든 FROZEN 세트가 있어야 실행됩니다.
- 조건 4개, 각 조건의 **원래 프롬프트·이미지 구성 그대로**: 원본만 `[target]` / 정상 이미지 `[normal, target]` / expert 점수 `[target]+PatchCore 점수 문장` / 결합 `[normal, target]+점수 문장`
- 위치 정확도(4지선다, 찍기 0.25), 원본만 대비 짝 비교(exact McNemar), 자산·결함·라벨 출처별 정확도
- 대략 문항 수 × 4조건 × 4모델 호출 (139문항이면 2,224회, 1시간 안팎)

이 노트북은 스크립트 출력(진행·에러)을 셀에 그대로 보여줍니다. 끊기면 위에서부터 다시 실행하면 완료된 문항은 재사용합니다.

In [ ]:
REPO = "arnold6444/lvlm_insplad"
REPO_REF = "main"
QA_FOLDER = "lvlm_insplad_qa/gold_v2_final"
LOC_FOLDER = "lvlm_insplad_qa/gold_v2_loc_mmad"                      # 박스 표시 노트북 결과
REFERENCE_FOLDER = "lvlm_insplad_results/gold_reference_v2"
VX_FOLDER = "lvlm_insplad_results/gold_v2_vision_expert_full"
COMBINED_FOLDER = "lvlm_insplad_results/gold_v2_reference_expert/predictions"
CONSISTENCY_FOLDER = "lvlm_insplad_results/gold_v2_consistency_metrics"
LOC_EVAL_FOLDER = "lvlm_insplad_results/gold_v2_loc_mmad_eval"

MODELS = ["qwen3_5_4b","internvl3_5_8b","minicpm_v_4_5","llava_onevision_7b"]
LOC_CONDITIONS = ["image_only","normal_ref","expert_score","combined"]
EXPERT = "patchcore"
CHECKPOINT_EVERY = 20
PUBLISH = True

UNSUP_ZIP_NAME = "unsupervised_anomaly_detection.zip"
UNSUP_ZIP_SIZE = 1_170_967_719
UNSUP_ZIP_PATH = ""   # 비우면 자동 검색
SUP_ZIP_NAME = "supervised_fault_classification.zip"
SUP_ZIP_SIZE = 874_753_167
SUP_ZIP_PATH = ""     # 비우면 자동 검색

In [ ]:
# Drive 연결 + repo clone + 패키지 (GPU는 B단계에서만 필요)
import base64, json, os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive")
QA_DIR = DRIVE_ROOT / QA_FOLDER
LOC_DIR = DRIVE_ROOT / LOC_FOLDER
REFERENCE_DIR = DRIVE_ROOT / REFERENCE_FOLDER
VX_PRED_DIR = DRIVE_ROOT / VX_FOLDER / "eval" / "predictions"
MERGED_MANIFEST = DRIVE_ROOT / VX_FOLDER / "merged_evidence" / "roi_manifest_all.csv"
COMBINED_DIR = DRIVE_ROOT / COMBINED_FOLDER
CONSISTENCY_DIR = DRIVE_ROOT / CONSISTENCY_FOLDER
LOC_EVAL_DIR = DRIVE_ROOT / LOC_EVAL_FOLDER

REPO_DIR = Path("/content/lvlm_insplad")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

def run(cmd, *, env=None, check=True, quiet_prefix=None):
    """Run a command and stream its output into this cell (Colab hides plain subprocess output)."""
    cmd = [str(x) for x in cmd]
    print("+", " ".join(cmd[:12]) + (" ..." if len(cmd) > 12 else ""), flush=True)
    p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        tail = (tail + [line])[-60:]
        if not quiet_prefix or not line.startswith(quiet_prefix):
            print(line, end="", flush=True)
    p.wait()
    if check and p.returncode:
        print("".join(tail))
        raise RuntimeError(f"command failed with exit code {p.returncode} (마지막 출력 위 참고)")
    return p.returncode

github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("Colab Secret GITHUB_TOKEN이 필요합니다.")
os.environ["GITHUB_TOKEN"] = github_token
try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_env = os.environ.copy()
git_env.update({"GIT_CONFIG_COUNT":"1", "GIT_CONFIG_KEY_0":"http.extraHeader",
                "GIT_CONFIG_VALUE_0":f"AUTHORIZATION: basic {auth}", "GIT_TERMINAL_PROMPT":"0"})
run(["git","clone","--depth","1","--branch",REPO_REF,f"https://github.com/{REPO}.git",REPO_DIR], env=git_env)
del auth, git_env
sys.path.insert(0, str(REPO_DIR / "src"))
sys.path.insert(0, str(REPO_DIR / "scripts"))

import pandas as pd
import torch
HAS_GPU = torch.cuda.is_available()
print("repo:", subprocess.check_output(["git","rev-parse","HEAD"], cwd=REPO_DIR, text=True).strip())
print("GPU:", torch.cuda.get_device_name(0) if HAS_GPU else "없음 (A단계만 가능)")
if HAS_GPU:
    run([sys.executable,"-m","pip","install","-q",
         "transformers==5.14.0","accelerate==1.14.0","sentencepiece==0.2.2",
         "pandas>=2.2","numpy>=1.26","pillow>=10","tqdm>=4.66","timm"])
IMAGE_ONLY_CSV = REPO_DIR / "results/latest/gold_v2_baseline/prediction_review.csv"

## A. 모순 지표 (새 추론 없음)

In [ ]:
run([sys.executable, REPO_DIR / "scripts/run_gold_v2_consistency_metrics.py",
     "--qa-dir", QA_DIR, "--image-only-csv", IMAGE_ONLY_CSV, "--reference-dir", REFERENCE_DIR,
     "--vx-pred-dir", VX_PRED_DIR, "--combined-dir", COMBINED_DIR,
     "--out-dir", CONSISTENCY_DIR, "--models", *MODELS])
cons = pd.read_csv(CONSISTENCY_DIR / "consistency_leaderboard.csv")
pd.set_option("display.width", 220); pd.set_option("display.max_rows", 300)
main_conditions = ["image_only","normal_reference","vx:patchcore:score","ref+patchcore:score"]
print("[주요 4조건]")
display(cons[cons.condition.isin(main_conditions)].round(3))
print("\n[모순 많은 순 상위 15]")
display(cons.sort_values("consistency_error", ascending=False).head(15).round(3))
meta = json.loads((CONSISTENCY_DIR / "run_metadata.json").read_text(encoding="utf-8"))
print("빠진 조건(예측 파일 불완전):", meta["skipped_incomplete"])

In [ ]:
# 검증: Q1 F1·Q2 macro-F1이 게시된 리더보드와 같은지 (주요 4조건)
L = REPO_DIR / "results/latest"
checks = {
    "image_only": pd.read_csv(L / "gold_v2_baseline/leaderboard.csv"),
    "normal_reference": pd.read_csv(L / "gold_v2_reference/leaderboard.csv").query("condition == 'normal_reference'"),
    "vx:patchcore:score": pd.read_csv(L / "gold_v2_vision_expert_full/leaderboard.csv").query("expert == 'patchcore' and condition == 'score'"),
    "ref+patchcore:score": pd.read_csv(L / "gold_v2_reference_expert/leaderboard.csv").query("expert == 'patchcore' and condition == 'score'"),
}
bad = []
for cond, ref in checks.items():
    ref = ref.set_index("model_key")
    for _, r in cons[cons.condition == cond].iterrows():
        for ours, theirs in (("q1_anomaly_f1","anomaly_f1"), ("q2_defect_macro_f1","defect_macro_f1")):
            if abs(r[ours] - float(ref.loc[r.model_key, theirs])) > 1e-4:
                bad.append((cond, r.model_key, ours, r[ours], float(ref.loc[r.model_key, theirs])))
if bad:
    display(pd.DataFrame(bad, columns=["condition","model","metric","computed","published"]))
    raise RuntimeError("게시된 리더보드와 다름 — 입력 확인")
print("검증 통과: 주요 4조건의 Q1·Q2 점수가 게시된 리더보드와 일치")

In [ ]:
if PUBLISH:
    run([sys.executable, REPO_DIR / "scripts/publish_results_to_github.py", "--out-dir", CONSISTENCY_DIR,
         "--repo", REPO, "--branch", REPO_REF, "--prefix", "gold_v2_consistency_metrics",
         "--files", "consistency_leaderboard.csv", "run_metadata.json"], env=os.environ.copy())
    print("GitHub: results/latest/gold_v2_consistency_metrics/")

## B. 새 위치 문항 평가 (L4 필요, 박스 표시 완료 후)

In [ ]:
summary_path = LOC_DIR / "build_summary.json"
if not summary_path.is_file():
    raise RuntimeError("위치 세트가 아직 없습니다 — 박스 표시 노트북을 먼저 끝내세요")
loc_summary = json.loads(summary_path.read_text(encoding="utf-8"))
if loc_summary.get("quality_status") != "FROZEN":
    raise RuntimeError(f"위치 세트가 아직 미완료입니다 (남은 이미지 {len(loc_summary.get('missing_annotation', []))}장)")
if not HAS_GPU:
    raise RuntimeError("B단계는 GPU(L4)가 필요합니다 — 런타임을 L4로 바꾸고 위에서부터 다시 실행")
LOC_QA = LOC_DIR / loc_summary["file"]

def find_zip(filename, expected_size, override=""):
    candidates = [Path(override)] if override else []
    candidates += [DRIVE_ROOT / filename, DRIVE_ROOT / "lvlm_insplad_data" / filename]
    for p in candidates:
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    print(f"{filename}: Drive 전체 검색 중 (몇 분 걸릴 수 있음)", flush=True)
    for p in DRIVE_ROOT.rglob(filename):
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    raise FileNotFoundError(f"{filename} ({expected_size} bytes) not found")

SUP_ZIP = find_zip(SUP_ZIP_NAME, SUP_ZIP_SIZE, SUP_ZIP_PATH)
UNSUP_ZIP = find_zip(UNSUP_ZIP_NAME, UNSUP_ZIP_SIZE, UNSUP_ZIP_PATH)
print("위치 문항:", LOC_QA.name, loc_summary["questions"], "문항", loc_summary["by_asset"])
print("예상 호출:", loc_summary["questions"] * len(LOC_CONDITIONS) * len(MODELS), "회")

In [ ]:
# 실행 — 끊기면 이 셀만 다시 실행해도 완료된 문항은 재사용
run([sys.executable, REPO_DIR / "scripts/run_gold_v2_loc_mmad_eval.py",
     "--loc-qa", LOC_QA, "--qa-dir", QA_DIR, "--sup-zip", SUP_ZIP, "--unsup-zip", UNSUP_ZIP,
     "--roi-manifest", MERGED_MANIFEST, "--out-dir", LOC_EVAL_DIR,
     "--models", *MODELS, "--conditions", *LOC_CONDITIONS, "--expert", EXPERT,
     "--checkpoint-every", str(CHECKPOINT_EVERY)])

In [ ]:
# 결과
lb = pd.read_csv(LOC_EVAL_DIR / "leaderboard.csv")
display(lb.round(3))
groups = pd.read_csv(LOC_EVAL_DIR / "metrics_by_group.csv")
for g in ("asset_category", "defect_gt", "label_basis"):
    print(f"\n[{g}별 위치 정확도]")
    display(groups[groups.group == g].pivot_table(index=["model_key","condition"], columns="value",
                                                   values="loc_accuracy").round(3))
status = json.loads((LOC_EVAL_DIR / "run_status.json").read_text(encoding="utf-8"))
print("row_errors 합계:", sum(s["row_errors"] for s in status), "(0이 아니면 실행 셀을 다시 실행)")

In [ ]:
if PUBLISH:
    run([sys.executable, REPO_DIR / "scripts/publish_results_to_github.py", "--out-dir", LOC_EVAL_DIR,
         "--repo", REPO, "--branch", REPO_REF, "--prefix", "gold_v2_loc_mmad_eval",
         "--files", "leaderboard.csv", "metrics_by_group.csv", "run_status.json", "run_metadata.json"],
        env=os.environ.copy())
    print("GitHub: results/latest/gold_v2_loc_mmad_eval/")